# 09 — Pretraining on Cosmopedia-v2

This notebook trains a **fresh SLM from scratch** on `HuggingFaceTB/smollm-corpus` (cosmopedia-v2 subset) — a dataset of synthetic educational content covering science, history, math, and everyday topics.

Why cosmopedia-v2?
- Designed specifically for training small language models
- Diverse topics → richer vocabulary and broader knowledge than TinyStories
- Clean, well-structured prose → stable training signal
- Used to train SmolLM-135M/360M (HuggingFace)

**Pipeline in this notebook:**
1. Stream & save 500k cosmopedia documents to Drive
2. Train a fresh BPE tokenizer on the cosmopedia vocabulary
3. Tokenize all splits into binary `.npy` files
4. Initialise a fresh GPT model
5. Pretrain for 50k steps (~2.5 hrs on T4)
6. Evaluate and generate sample completions

**Expected outcome:** A small but capable model that can discuss educational topics, write short explanations, and answer simple factual questions.

## Setup

In [ ]:
# ── Environment setup ─────────────────────────────────────────────────────
# Run this cell first every Colab session.
import os, sys

if os.path.exists("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

sys.path.insert(0, os.path.abspath(".."))

import config as cfg
cfg.make_dirs()
cfg.print_config()

In [ ]:
# !pip install datasets tokenizers tqdm torch transformers

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Step 1 — Stream & save cosmopedia-v2

We use `streaming=True` so the full 28B-token dataset is never downloaded — we stop after `cfg.COSMO_DATA_SIZE` examples.

The last 10k examples are held out for validation (5k) and test (5k).

**Expected time:** 10–20 minutes depending on Colab network speed.

In [ ]:
from datasets import load_dataset
from tqdm import tqdm
import config as cfg

# Total examples to pull: train + val + test
TOTAL = cfg.COSMO_DATA_SIZE + cfg.COSMO_VAL_SIZE + cfg.COSMO_TEST_SIZE

print(f"Streaming {TOTAL:,} examples from {cfg.COSMO_DATASET_NAME} / {cfg.COSMO_DATASET_SUBSET}")
print(f"  train: {cfg.COSMO_DATA_SIZE:,}  val: {cfg.COSMO_VAL_SIZE:,}  test: {cfg.COSMO_TEST_SIZE:,}")

dataset = load_dataset(
    cfg.COSMO_DATASET_NAME,
    cfg.COSMO_DATASET_SUBSET,
    split="train",
    streaming=True,   # critical — avoids downloading 28B tokens
)

train_examples, val_examples, test_examples = [], [], []

for i, example in enumerate(tqdm(dataset, total=TOTAL, desc="Streaming")):
    if i >= TOTAL:
        break
    if i < cfg.COSMO_DATA_SIZE:
        train_examples.append(example["text"])
    elif i < cfg.COSMO_DATA_SIZE + cfg.COSMO_VAL_SIZE:
        val_examples.append(example["text"])
    else:
        test_examples.append(example["text"])

print(f"\nCollected  train={len(train_examples):,}  val={len(val_examples):,}  test={len(test_examples):,}")

In [ ]:
# Peek at a sample to understand the data format
print(train_examples[0][:800])
print("...")

In [ ]:
# Save splits to text files (one document per line)
# We replace internal newlines with spaces so each line = one document,
# which is what the tokenizer trainer and tokenize_and_save expect.

def save_split(examples, path):
    with open(path, "w", encoding="utf-8") as f:
        for text in tqdm(examples, desc=f"Saving {os.path.basename(path)}"):
            f.write(text.replace("\n", " ").strip() + "\n")
    print(f"Saved {len(examples):,} examples → {path}")

save_split(train_examples, cfg.COSMO_TRAIN_TXT)
save_split(val_examples,   cfg.COSMO_VAL_TXT)
save_split(test_examples,  cfg.COSMO_TEST_TXT)

# Free memory — we no longer need the raw text lists
del train_examples, val_examples, test_examples

## Step 2 — Train a fresh BPE tokenizer

We train a new tokenizer on the cosmopedia training text rather than reusing the TinyStories tokenizer.

**Why retrain?**  
The TinyStories tokenizer was trained on simple children's story vocabulary. Cosmopedia covers science, history, and math — it has many words (e.g. "photosynthesis", "coefficient", "legislature") that would be split into many subword tokens by the TinyStories tokenizer, making the model less efficient.

A tokenizer trained on cosmopedia vocabulary will represent this domain more compactly.

**Expected time:** ~5 minutes.

In [ ]:
from tokenizer.preprocess import train_tokenizer
import config as cfg

tokenizer = train_tokenizer(
    train_file=cfg.COSMO_TRAIN_TXT,
    save_dir=cfg.COSMO_TOKENIZER_DIR,
    vocab_size=cfg.VOCAB_SIZE,   # keep the same vocab size: 8192
)

print(f"Vocab size: {tokenizer.get_vocab_size()}")
print(f"EOS token id: {tokenizer.token_to_id('</s>')}")

In [ ]:
# Quick sanity check — see how the tokenizer handles domain-specific words
test_words = ["photosynthesis", "coefficient", "democracy", "Once upon a time"]
for w in test_words:
    ids = tokenizer.encode(w).ids
    tokens = [tokenizer.id_to_token(i) for i in ids]
    print(f"  {w!r:30s} → {len(ids)} tokens: {tokens}")

## Step 3 — Tokenize splits

Convert all three text splits into flat `uint16` numpy arrays.  
These are memory-mapped during training so the full file never loads into RAM.

**Expected time:** 20–40 minutes for 500k documents.

In [ ]:
# Override the default paths inside preprocess.py to point at cosmopedia paths.
# We do this by temporarily patching config so tokenize_and_save reads the
# right txt/npy paths — cleaner than duplicating the function.

import config as cfg
from tokenizer.preprocess import load_tokenizer

# Patch config paths for this run
_orig = {
    "TRAIN_TXT": cfg.TRAIN_TXT, "VAL_TXT": cfg.VAL_TXT, "TEST_TXT": cfg.TEST_TXT,
    "TRAIN_TOKENS": cfg.TRAIN_TOKENS, "VAL_TOKENS": cfg.VAL_TOKENS, "TEST_TOKENS": cfg.TEST_TOKENS,
}

cfg.TRAIN_TXT    = cfg.COSMO_TRAIN_TXT;    cfg.TRAIN_TOKENS = cfg.COSMO_TRAIN_TOKENS
cfg.VAL_TXT      = cfg.COSMO_VAL_TXT;      cfg.VAL_TOKENS   = cfg.COSMO_VAL_TOKENS
cfg.TEST_TXT     = cfg.COSMO_TEST_TXT;     cfg.TEST_TOKENS  = cfg.COSMO_TEST_TOKENS

from tokenizer.preprocess import tokenize_and_save

tokenizer = load_tokenizer(cfg.COSMO_TOKENIZER_VOCAB, cfg.COSMO_TOKENIZER_MERGES)

for split in ["train", "val", "test"]:
    tokenize_and_save(split, tokenizer, batch_size=4000)

# Restore original config paths
for k, v in _orig.items():
    setattr(cfg, k, v)

In [ ]:
import numpy as np

# Verify token counts
for name, path in [("train", cfg.COSMO_TRAIN_TOKENS),
                   ("val",   cfg.COSMO_VAL_TOKENS),
                   ("test",  cfg.COSMO_TEST_TOKENS)]:
    arr = np.memmap(path, dtype=np.uint16, mode="r")
    print(f"  {name}: {len(arr):>12,} tokens  ({len(arr)/1e6:.1f}M)")

## Step 4 — Initialise the model

Same GPT architecture as the TinyStories run — 8 layers, 8 heads, 512-dim, ~25M parameters.  
All weights are randomly initialised — this is a completely fresh model.

In [ ]:
import torch
from model import GPT, GPTConfig
import config as cfg

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Training on:", device)

config = GPTConfig()   # reads defaults from config.py
model  = GPT(config).to(device)

print(f"Parameters: {model.num_parameters() / 1e6:.2f}M")
print(f"Config: {config.__dict__}")

In [ ]:
# Quick forward-pass check before committing to a long training run
x_test = torch.randint(0, config.vocab_size, (2, config.block_size)).to(device)
with torch.no_grad():
    logits = model(x_test)
print("Forward pass OK — logits shape:", logits.shape)  # expect (2, 256, 8192)

## Step 5 — Pretraining

Training details:
- **Optimizer:** AdamW with `lr=3e-4`, `weight_decay=0.1`
- **Steps:** 50,000 (~2.5 hrs on T4)
- **Warmup:** 500 steps linear LR warmup → cosine decay
- **Grad clip:** 1.0 (prevents gradient spikes on harder text)
- **Checkpoint:** saved every 1,000 steps → auto-resumes if Colab disconnects

**Expected loss curve:**
- Step 0: ~9.0 (random model, log(8192))
- Step 5k: ~4.5–5.0
- Step 20k: ~3.5–4.0
- Step 50k: ~3.0–3.5 (cosmopedia is harder than TinyStories — lower PPL takes more steps)

In [ ]:
import os
import math
import torch
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm
import config as cfg

# ── Data loaders ──────────────────────────────────────────────────────────
train_tokens = np.memmap(cfg.COSMO_TRAIN_TOKENS, dtype=np.uint16, mode="r")
val_tokens   = np.memmap(cfg.COSMO_VAL_TOKENS,   dtype=np.uint16, mode="r")

block_size = config.block_size
batch_size = cfg.COSMO_BATCH_SIZE

def get_batch(data):
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x  = torch.stack([torch.from_numpy(data[i   : i+block_size  ].astype(np.int64)) for i in ix])
    y  = torch.stack([torch.from_numpy(data[i+1 : i+block_size+1].astype(np.int64)) for i in ix])
    return x.to(device), y.to(device)

# ── LR schedule: linear warmup → cosine decay ────────────────────────────
def get_lr(step):
    # Linear warmup for the first COSMO_WARMUP_STEPS steps
    if step < cfg.COSMO_WARMUP_STEPS:
        return cfg.COSMO_LR * (step + 1) / cfg.COSMO_WARMUP_STEPS
    # Cosine decay from warmup end to max steps
    progress = (step - cfg.COSMO_WARMUP_STEPS) / (cfg.COSMO_MAX_STEPS - cfg.COSMO_WARMUP_STEPS)
    return cfg.COSMO_LR * 0.5 * (1.0 + math.cos(math.pi * progress))

# ── Optimizer ─────────────────────────────────────────────────────────────
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=cfg.COSMO_LR,
    weight_decay=cfg.COSMO_WEIGHT_DECAY,
)

# ── Validation loss estimate ───────────────────────────────────────────────
@torch.no_grad()
def estimate_val_loss():
    model.eval()
    losses = []
    for _ in range(cfg.COSMO_EVAL_BATCHES):
        x, y = get_batch(val_tokens)
        logits = model(x)
        loss = F.cross_entropy(logits.view(-1, config.vocab_size), y.view(-1))
        losses.append(loss.item())
    model.train()
    return sum(losses) / len(losses)

# ── Resume from checkpoint ────────────────────────────────────────────────
start_step = 0
if os.path.exists(cfg.COSMO_CKPT):
    print("Resuming from checkpoint:", cfg.COSMO_CKPT)
    ckpt = torch.load(cfg.COSMO_CKPT, map_location=device)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    start_step = ckpt["step"]
    print(f"  → resuming from step {start_step}")

print(f"Training for {cfg.COSMO_MAX_STEPS - start_step:,} steps  "
      f"(steps {start_step} → {cfg.COSMO_MAX_STEPS})")

In [ ]:
%%time
# ── Training loop ─────────────────────────────────────────────────────────
model.train()

for step in range(start_step, cfg.COSMO_MAX_STEPS):

    # Update learning rate (warmup → cosine decay)
    lr = get_lr(step)
    for param_group in optimizer.param_groups:
        param_group["lr"] = lr

    x, y = get_batch(train_tokens)

    logits = model(x)
    loss   = F.cross_entropy(logits.view(-1, config.vocab_size), y.view(-1))

    optimizer.zero_grad()
    loss.backward()

    # Gradient clipping — prevents large gradient spikes on hard examples
    if cfg.COSMO_GRAD_CLIP > 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.COSMO_GRAD_CLIP)

    optimizer.step()

    # ── Logging ──
    if step % 50 == 0:
        print(f"step {step:>6} | lr {lr:.2e} | train loss {loss.item():.4f}")

    # ── Validation ──
    if step % cfg.COSMO_EVAL_INTERVAL == 0 and step > 0:
        val_loss = estimate_val_loss()
        print(f"step {step:>6} | val loss {val_loss:.4f} | val ppl {math.exp(val_loss):.1f}")

    # ── Checkpoint ──
    if step % cfg.COSMO_SAVE_INTERVAL == 0 and step > 0:
        torch.save(
            {"model": model.state_dict(), "optimizer": optimizer.state_dict(), "step": step},
            cfg.COSMO_CKPT,
        )

# ── Save final weights ────────────────────────────────────────────────────
torch.save(model.state_dict(), cfg.COSMO_FINAL_CKPT)
print(f"\nTraining complete. Model saved → {cfg.COSMO_FINAL_CKPT}")

## Step 6 — Evaluate

Compute perplexity on the val and test splits.

In [ ]:
import math

test_tokens = np.memmap(cfg.COSMO_TEST_TOKENS, dtype=np.uint16, mode="r")

@torch.no_grad()
def eval_split(data, num_batches=100):
    model.eval()
    losses = []
    for _ in range(num_batches):
        x, y = get_batch(data)
        logits = model(x)
        loss = F.cross_entropy(logits.view(-1, config.vocab_size), y.view(-1))
        losses.append(loss.item())
    mean_loss = sum(losses) / len(losses)
    return mean_loss, math.exp(mean_loss)

val_loss,  val_ppl  = eval_split(val_tokens)
test_loss, test_ppl = eval_split(test_tokens)

print(f"Val  — loss: {val_loss:.4f}  ppl: {val_ppl:.2f}")
print(f"Test — loss: {test_loss:.4f}  ppl: {test_ppl:.2f}")

## Step 7 — Generate samples

Test the model with a few prompts. Because cosmopedia covers educational content, try prompts that ask for explanations.

In [ ]:
from tokenizer.preprocess import load_tokenizer
from generation.sampler import generate, generate_kv, encode_prompt
import textwrap
import config as cfg

tokenizer = load_tokenizer(cfg.COSMO_TOKENIZER_VOCAB, cfg.COSMO_TOKENIZER_MERGES)

model.eval()

prompts = [
    "Photosynthesis is the process by which",
    "The French Revolution began in 1789 when",
    "To solve a quadratic equation, you can",
    "The water cycle consists of several stages:",
    "Democracy is a system of government where",
]

for prompt in prompts:
    x   = encode_prompt(prompt, tokenizer, device)
    out = generate(
        model, x, tokenizer,
        block_size=config.block_size,
        max_new_tokens=150,
        temperature=0.8,
        top_k=50,
        top_p=0.9,
        repetition_penalty=1.1,
    )
    text = tokenizer.decode(out[0].tolist())
    print(f"\nPROMPT: {prompt}")
    print(textwrap.fill(text, 80))
    print("─" * 80)

In [ ]:
# Compare generation speed: with vs without KV cache
from model import GPTWithKVCache

model_kv = GPTWithKVCache(config).to(device)
model_kv.load_state_dict(torch.load(cfg.COSMO_FINAL_CKPT, map_location=device))
model_kv.eval()

prompt = "The speed of light is approximately"
x      = encode_prompt(prompt, tokenizer, device)

In [ ]:
%%time
out = generate(model, x, tokenizer, block_size=config.block_size, max_new_tokens=200)
print("Without KV cache:")
print(textwrap.fill(tokenizer.decode(out[0].tolist()), 80))

In [ ]:
%%time
out_kv = generate_kv(model_kv, x, tokenizer, block_size=config.block_size, max_new_tokens=200)
print("With KV cache:")
print(textwrap.fill(tokenizer.decode(out_kv[0].tolist()), 80))

## Next steps

Now that you have a pretrained cosmopedia model, the next steps follow the same pipeline as TinyStories:

1. **SFT** — Fine-tune on instruction-formatted cosmopedia data (notebook 07 pattern, swap dataset)
2. **DPO** — Generate preference pairs and run DPO (notebook 08 pattern)
3. **Push to HuggingFace Hub** — Set `cfg.HF_MODEL_REPO` and push tokenizer + weights

To push the tokenizer:
```python
from huggingface_hub import HfApi
api = HfApi()
api.upload_folder(folder_path=cfg.COSMO_TOKENIZER_DIR, repo_id=cfg.HF_TOKENIZER_REPO)
```